In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import csv
import pickle as pkl
import pickle
import copy
import re
import random
import matplotlib.pyplot as plt
import itertools
import json
import openai
import time
import os


!pip install ipython-autotime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 13.1 MB/s eta 0:00:00


In [ ]:

from google.colab import userdata
my_secret_key = userdata.get('API_KEY')

# wandb_secret_key = userdata.get('WANDB_KEY')

if my_secret_key:
  print("Token retrieved successfully.")
else:
  print("Token not found in Colab Secrets.")

from openai import OpenAI

client = OpenAI(
    # This is the default and can be omitted
    api_key = my_secret_key,
)


import asyncio
from openai import AsyncOpenAI

async_client = AsyncOpenAI(api_key = my_secret_key,
)  # make sure this is your actual key

Token retrieved successfully.


In [46]:
!rm -rf LLM4BEAR
!git clone --depth 1 --filter=blob:none --sparse https://github.com/anon5159753/LLM4BEAR.git
!cd LLM4BEAR && git sparse-checkout set "4_Bundle Generation"

Cloning into 'LLM4BEAR'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 59 (delta 6), reused 38 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 37.26 KiB | 1.01 MiB/s, done.
Resolving deltas: 100% (6/6), done.
remote: Enumerating objects: 1, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1 (delta 0), reused 1 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (1/1), 67 bytes | 67.00 KiB/s, done.
remote: Enumerating objects: 593, done.
remote: Counting objects: 100% (593/593), done.
remote: Compressing objects: 100% (562/562), done.
remote: Total 593 (delta 283), reused 207 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (593/593), 78.21 MiB | 8.07 MiB/s, done.
Resolving deltas: 100% (283/283), done.
Updating files: 100% (598/598), done.


In [ ]:

import os

path = "/content/drive/MyDrive/evaluating_bundles/"

try:
    os.makedirs(path, exist_ok=True)
    print(f"Successfully created: {path}")
except Exception as e:
    print(f"An error occurred: {e}")



Successfully created: /content/drive/MyDrive/evaluating_bundles/


In [ ]:
def infer_intent_prompt(input_string):
    """
    Refined prompt to infer specific e-commerce intent from a product list.
    """
    output_format = f"""
Analyze the following list of products and infer the specific underlying intent or use-case they represent:

{input_string}
**## TASK:**
1. **Analyze:** Look for logical connections (e.g., a specific hobby, a repair task, a seasonal event, or a targeted lifestyle need).
2. **Think Step-by-Step:** Briefly reason about why these items are grouped together.
3. **Summarize Intent:** Create a hyper-specific 3-4 word theme.

**## CONSTRAINTS for 'intent':**
- Must be **specific** (e.g., "Organic Backyard Tomato Gardening" instead of "Gardening Supplies").
- Avoid broad categories (e.g., do not use "Electronics" or "Kitchenware").
- **Strictly Prohibited:** Do not include filler phrases like "and accessories," "and tools," "related items," or "and equipment."

**## OUTPUT FORMAT:**
Print your reasoning first. Then, print the separator string `===JSON_START===` on a new line. Finally, provide a single JSON object:

```json
{{
  "intent": "string (3-4 words max)"
}}"""


    return output_format

In [ ]:
def input_strings(item_list):
    l = len(item_list)
    string_list = []
    for i in range(l):
        items = item_list[i]
        item_str = "\n".join([f"{i + 1}. {title}" for i, title in enumerate(items)])

        string_list.append(f"Intent: \nBundle Items:\n{item_str}\n")
    return string_list

def extract_inferred_intent(response):
    json_schema = extract_json_simple_replace(response)
    if json_schema and 'intent' in json_schema:
        return str(json_schema['intent']).strip()
    return None


def extract_bundle_score(response):
    json_schema = extract_json_simple_replace(response)
    if json_schema is not None:
        try:
            given_score = float(json_schema['score'])
        except:
            given_score = None
    else:
        given_score = None

    return given_score

def extract_bundle_verdict(response, consideration):
    json_schema = extract_json_simple_replace(response)
    if json_schema is not None:
        try:
            if consideration == "1-2":
                given_verdict_1 = json_schema['is_poor_quality_bundle']
                given_verdict_2 = json_schema['is_acceptable_quality_bundle']
                given_verdict = [given_verdict_1.lower(), given_verdict_2.lower()]
            elif consideration == "3":
                given_verdict_1 = json_schema['needs_improvement_bundle']
                given_verdict_2 = json_schema['is_good_quality_bundle']
                given_verdict = [given_verdict_1.lower(), given_verdict_2.lower()]
            elif consideration == "4-5":
                given_verdict_1 = json_schema['needs_improvement_bundle']
                given_verdict_2 = json_schema['is_high_quality_bundle']
                given_verdict = [given_verdict_1.lower(), given_verdict_2.lower()]
            else:
                given_verdict = None
        except:
            given_verdict = None
    else:
        given_verdict = None

    return given_verdict

def extract_json_simple_replace(response_text):
    if response_text is None:
        return None

    try:
        # 1. Use a case-insensitive split or check for the separator
        if "===JSON_START===" not in response_text:
            # Fallback: Try to find the first '{' anyway
            json_part = response_text
        else:
            json_part = response_text.split("===JSON_START===")[1]

        # 2. Find the boundaries
        first_brace = json_part.find('{')
        last_brace = json_part.rfind('}')

        if first_brace == -1 or last_brace == -1:
            return None

        # 3. Extract and clean
        json_string = json_part[first_brace : last_brace + 1].strip()

        # 4. Parse
        return json.loads(json_string)

    except Exception as e:
        print(f"Extraction error: {e}")
        return None


In [ ]:
async def single_request(user, system=None, seed_value=None):

    if system:
        message = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    else:
        message = [{"role": "user", "content": user}]

    # Reimplemented the robust retry loop with exponential backoff
    for delay_secs in (2**x for x in range(0, 3)):
        try:
            response = await async_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=message,
                temperature=0,
                max_tokens=1500,
                seed=seed_value
            )
            return response.choices[0].message.content.strip()
        except openai.OpenAIError as e:
            randomness_collision_avoidance = random.randint(0, 1000) / 1000.0
            sleep_dur = delay_secs + randomness_collision_avoidance
            print(f"Error: {e}. Retrying in {round(sleep_dur, 2)} seconds.")
            await asyncio.sleep(sleep_dur)

    # Return None if all retries fail
    return None


async def openai_request(prompts, system=None, batch_size=128, delay=0):

    results = []

    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        tasks = [
            single_request(d["prompts"], system=system, seed_value=42)
            for j, d in enumerate(batch)
        ]

        batch_results = await asyncio.gather(*tasks)
        results.extend(batch_results)
        print(f"✅ Sending batch {i // batch_size + 1} — sleeping for {delay}s...\n")
        await asyncio.sleep(delay)

    return results


async def separate_consideration_scores(char, initial_prompt, sample_data, json_addition, constant_metrics = "", consideration = "", save = ""):

    if consideration == "":
        return

    else:

        prompt_list = [{"prompts": data + "\n" + json_addition} for data in sample_data]

        print(initial_prompt + constant_metrics)
        print(prompt_list[0]["prompts"])

        responses = await openai_request(prompt_list, initial_prompt + constant_metrics)

        if consideration == "":
            scores = [extract_bundle_score(i) for i in responses]

            verdicts = None

        elif consideration == "1-2" or consideration == "3" or consideration == "4-5":
            verdicts = [extract_bundle_verdict(i, consideration) for i in responses]

            scores = [extract_bundle_score(i) for i in responses]


        filename = f"/content/drive/MyDrive/evaluating_bundles/{save}{char}.pkl"

        with open(filename, 'wb') as f:
            pickle.dump([responses, verdicts, scores], f)

        return responses, verdicts, scores



async def intent_evaluation_module(charizards, prompts, input_strings):

    intent_prompt_list = [{"prompts": infer_intent_prompt(data)} for data in input_strings]

    print(intent_prompt_list[0]["prompts"])

    intent_responses = await openai_request(intent_prompt_list)

    print(intent_responses[0])

    inferred_data = []

    for i in range(len(intent_responses)):
        raw_response = intent_responses[i]
        intent_text = extract_inferred_intent(raw_response)

        original_text = input_strings[i]

        # If intent exists, replace the placeholder.
        # If it's None, we just keep the original text as is.
        if intent_text:
            augmented_text = original_text.replace("Intent:", f"Intent: {intent_text}")
        else:
            augmented_text = original_text

        inferred_data.append(augmented_text)

    bad_responses, bad_verdicts, bad_scores = await separate_consideration_scores(char= charizards[0],
                                  initial_prompt= prompts[0],
                                  sample_data = inferred_data,
                                  json_addition = json_bad,
                                  constant_metrics = adding_metrics,
                                  consideration = "1-2",
                                                                                  save="intent_")


    middle_responses, middle_verdicts, middle_scores = await separate_consideration_scores(char= charizards[1],
                                  initial_prompt= prompts[1],
                                  sample_data = inferred_data,
                                  json_addition = json_middle,
                                  constant_metrics = adding_metrics,
                                  consideration = "3",
                                                                                           save="intent_")

    good_responses, good_verdicts, good_scores = await separate_consideration_scores(char= charizards[2],
                                  initial_prompt= prompts[2],
                                  sample_data = inferred_data,
                                  json_addition = json_good,
                                  constant_metrics = adding_metrics,
                                  consideration = "4-5",
                                                                                     save="intent_")


    return [bad_responses, bad_verdicts, bad_scores], [middle_responses, middle_verdicts, middle_scores], [good_responses, good_verdicts, good_scores]



In [ ]:
electronic_charizards = [
                  "bad_evaluator_electronic_importance_no_experts", "middle_evaluator_electronic_importance_no_experts", "good_evaluator_electronic_importance_no_experts",              ]

electronic_charizard_indices = [1, 1, 2]

electronic_prompts_to_test = []

for char in electronic_charizards:

    # filename = f"/content/drive/MyDrive/EGPO/final_prompts/Refined_{char}.pkl"
    filename = f"/content/LLM4BEAR/4_Bundle Generation/final_prompts/Refined_{char}.pkl"

    with open(filename, 'rb') as f:
        top_3_prompts, _ = pickle.load(f)

    electronic_prompts_to_test.append(top_3_prompts)


electronic_bad_prompt = electronic_prompts_to_test[0][1]
electronic_middle_prompt = electronic_prompts_to_test[1][1]
electronic_good_prompt = electronic_prompts_to_test[2][2]

electronic_prompts = [electronic_bad_prompt, electronic_middle_prompt, electronic_good_prompt]

clothing_charizards = [
                  "bad_evaluator_clothing_importance_no_experts", "middle_evaluator_clothing_importance_no_experts", "good_evaluator_clothing_importance_no_experts",              ]

clothing_charizard_indices = [2, 2, 0]

clothing_prompts_to_test = []

for char in clothing_charizards:

    # filename = f"/content/drive/MyDrive/EGPO/final_prompts/Refined_{char}.pkl"
    filename = f"/content/LLM4BEAR/4_Bundle Generation/final_prompts/Refined_{char}.pkl"


    with open(filename, 'rb') as f:
        top_3_prompts, _ = pickle.load(f)

    clothing_prompts_to_test.append(top_3_prompts)


clothing_bad_prompt = clothing_prompts_to_test[0][2]
clothing_middle_prompt = clothing_prompts_to_test[1][2]
clothing_good_prompt = clothing_prompts_to_test[2][0]

clothing_prompts = [clothing_bad_prompt, clothing_middle_prompt, clothing_good_prompt]


food_charizards = [
                  "bad_evaluator_food_importance_no_experts", "middle_evaluator_food_importance_no_experts", "good_evaluator_food_importance_no_experts"              ]

food_charizard_indices = [0, 2, 0]

food_prompts_to_test = []

for char in food_charizards:

    # filename = f"/content/drive/MyDrive/EGPO/final_prompts/Refined_{char}.pkl"
    filename = f"/content/LLM4BEAR/4_Bundle Generation/final_prompts/Refined_{char}.pkl"

    with open(filename, 'rb') as f:
        top_3_prompts, _ = pickle.load(f)

    food_prompts_to_test.append(top_3_prompts)


food_bad_prompt = food_prompts_to_test[0][0]
food_middle_prompt = food_prompts_to_test[1][2]
food_good_prompt = food_prompts_to_test[2][0]

food_prompts = [food_bad_prompt, food_middle_prompt, food_good_prompt]



json_bad = "After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions from Part 1. Do not include any other text after the separator.\n"\
            "**JSON Schema:**\n"\
            "```json\n"\
            "{{\n"\
            "is_poor_quality_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
            "is_acceptable_quality_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
            "score: float, bundle quality out of 5.\n"\
            "}}"


json_middle = "After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions from Part 1. Do not include any other text after the separator.\n"\
            "**JSON Schema:**\n"\
            "```json\n"\
            "{{\n"\
            "needs_improvement_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
            "is_good_quality_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
            "score: float, bundle quality out of 5.\n"\
            "}}"

json_good = "After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions from Part 1. Do not include any other text after the separator.\n"\
            "**JSON Schema:**\n"\
            "```json\n"\
            "{{\n"\
            "needs_improvement_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
            "is_high_quality_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
            "score: float, bundle quality out of 5.\n"\
            "}}"


adding_metrics = "\nFunctionality Integration: Describe how a user would utilize this collection of items to achieve their primary goal. Considering the entire workflow, is this a complete and logical set of items for the task, or is there an irrelevant or missing item? \n"\
              "Similarity: What is the common theme or category that connects these items?\n"\
              "Complementarity: Are these items more valuable together than they would be if sold separately? Does the presence of one item create a clear reason to buy the other(s)?\n"\
              "Diversity: Does the variety of items in this bundle cater to a broad set of related needs for a single user, or does the mix of items seem unfocused and random?\n"


In [ ]:
def load_pkl(file_path):
    """Safely loads a pickle file."""
    try:
        with open(file_path, 'rb') as f:
            return pkl.load(f)
    except FileNotFoundError:
        print(f"❌ ERROR: File not found at {file_path}")
        return None
    except Exception as e:
        print(f"❌ ERROR loading pickle file: {e}")
        return None


def clean_string(string):
    string = string.replace("<pad>", "").replace("</s>", "").strip()
    string = string.replace("[SEP]", " [SEP]")
    temp_split = string.split()
    temp_split = [i for i in temp_split if i != "p"]
    string = " ".join(temp_split)
    if string[-5:] == "[SEP]":
        string = string[:-5]
    string = string.replace("p", "product")
    string = string.strip()
    return string

def form_bundles(string):

    bundle_strings = string.split(' [SEP] ')

    pred_bundles = []

    for i in bundle_strings:
        holder = []
        individual = i.split(' ')

        for j in individual:

            holder.append(j)

        pred_bundles.append(holder)

    return pred_bundles

def reinforce_dict(dict):
    keys = list(dict.keys())
    values = list(dict.values())
    new_dict = {}

    for i in range(len(keys)):
        key = keys[i]
        value = values[i]
        another_dict = {}

        for j in range(len(value)):

            sub_key = f"bundle{j+1}"

            another_dict[sub_key] = value[j]



        new_dict[key] = another_dict

    return new_dict

def remove_empty_dict_entries(data_dict):
    """
    Removes entries from a dictionary where the value is an empty dictionary ({}).

    Args:
        data_dict (dict): The dictionary to be cleaned.

    Returns:
        dict: A new dictionary with the empty entries removed.
    """
    # Use dictionary comprehension to filter out items where the value (v) is falsy.
    # An empty dictionary {} evaluates to False in a boolean context.
    # This is a concise and standard Python technique.
    cleaned_dict = {k: v for k, v in data_dict.items() if v}

    return cleaned_dict

import re


def convert_products_to_indices(product_names):
    """
    Converts product identifiers (e.g., 'product2', '2', or 2)
    into zero-based integer indices (N-1).
    """
    indices = []
    # Regex to find the digits anywhere in the string
    pattern = re.compile(r'(\d+)')

    for name in product_names:
        # Convert to string to handle raw integers like 2 or 4
        name_str = str(name).strip()

        match = pattern.search(name_str)
        if match:
            # Extract the number
            product_number = int(match.group(1))
            # Convert to zero-based index
            index = product_number - 1
            indices.append(index)

    return indices



# I only need the keys to make the dictionaries. We are using the same sessions for training, validation and testing between BundleRec and LLM4BEAR.

In [ ]:
# electronic_bundlerec_path = '/content/drive/My Drive/baselines/bundlerec/electronic/'
# clothing_bundlerec_path = '/content/drive/My Drive/baselines/bundlerec/clothing/'
# food_bundlerec_path = '/content/drive/My Drive/AICL/baselines/bundlerec/food/'

# electronic_llm4bear_path = '/content/drive/My Drive/baselines/llm4bear/electronic/'
# clothing_llm4bear_path = '/content/drive/My Drive/baselines/llm4bear/clothing/'
# food_llm4bear_path = '/content/drive/My Drive/baselines/llm4bear/food/'

electronic_bundlerec_path = '/content/LLM4BEAR/4_Bundle Generation/baselines/bundlerec/electronic/'
clothing_bundlerec_path = '/content/LLM4BEAR/4_Bundle Generation/baselines/bundlerec/clothing/'
food_bundlerec_path = '/content/LLM4BEAR/4_Bundle Generation/baselines/bundlerec/food/'

electronic_llm4bear_path = '/content/LLM4BEAR/4_Bundle Generation/baselines/llm4bear/electronic/'
clothing_llm4bear_path = '/content/LLM4BEAR/4_Bundle Generation/baselines/llm4bear/clothing/'
food_llm4bear_path = '/content/LLM4BEAR/4_Bundle Generation/baselines/llm4bear/food/'


six_paths = [electronic_bundlerec_path, clothing_bundlerec_path, food_bundlerec_path, electronic_llm4bear_path, clothing_llm4bear_path, food_llm4bear_path]


baseline_names = ["gpt-4o-mini", "gpt-4.1-mini","gemini", "claude", "llama", "mistral"]


test_p_data_path = "test_p_data.npy"

electronic_bundlerec_test_p_data = np.load('/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/electronic/' + test_p_data_path, allow_pickle=True).tolist()
electronic_test_keys = list(electronic_bundlerec_test_p_data.keys())

clothing_bundlerec_test_p_data = np.load('/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/clothing/' + test_p_data_path, allow_pickle=True).tolist()
clothing_test_keys = list(clothing_bundlerec_test_p_data.keys())

food_bundlerec_test_p_data = np.load('/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/food/' + test_p_data_path, allow_pickle=True).tolist()
food_test_keys = list(food_bundlerec_test_p_data.keys())

keys = [electronic_test_keys, clothing_test_keys, food_test_keys]


zero_dicts = []
few_dicts = []

zero_lists = []
few_lists = []

for baseline in baseline_names:
    for i in range(6):
        zero_responses = load_pkl(os.path.join(six_paths[i], f'{baseline}_zero_shot_responses.pkl'))
        few_responses = load_pkl(os.path.join(six_paths[i], f'{baseline}_few_shot_responses.pkl'))

        zero_results = [extract_json_simple_replace(i) for i in zero_responses]
        few_results = [extract_json_simple_replace(i) for i in few_responses]

        zero_lists.append(zero_results)
        few_lists.append(few_results)

        zero_response_dict = dict(zip(keys[i%3], zero_results))
        few_response_dict = dict(zip(keys[i%3], few_results))


        zero_dicts.append(zero_response_dict)
        few_dicts.append(few_response_dict)


base_models = ["4o-mini", '4.1-mini']


other_dicts = []
other_lists = []

for other in ["4o-mini", "4.1-mini"]: # or however you named each

    # AICL_brec_electronic = remove_empty_dict_entries(np.load(f"/content/drive/My Drive/baselines/AICL/results/bundlerec/electronic/{other}_bundle_res.npy", allow_pickle=True).tolist())
    # AICL_brec_clothing = remove_empty_dict_entries(np.load(f"/content/drive/My Drive/baselines/AICL/results/bundlerec/clothing/{other}_bundle_res.npy", allow_pickle=True).tolist())
    # AICL_brec_food = remove_empty_dict_entries(np.load(f"/content/drive/My Drive/baselines/AICL/results/bundlerec/food/{other}_bundle_res.npy", allow_pickle=True).tolist())

    # AICL_bear_electronic = remove_empty_dict_entries(np.load(f"/content/drive/My Drive/baselines/AICL/results/llm4bear/electronic/{other}_bundle_res.npy", allow_pickle=True).tolist())
    # AICL_bear_clothing = remove_empty_dict_entries(np.load(f"/content/drive/My Drive/baselines/AICL/results/llm4bear/clothing/{other}_bundle_res.npy", allow_pickle=True).tolist())
    # AICL_bear_food = remove_empty_dict_entries(np.load(f"/content/drive/My Drive/baselines/AICL/results/llm4bear/food/{other}_bundle_res.npy", allow_pickle=True).tolist())


    for i in range(6):

        AICL = remove_empty_dict_entries(np.load(six_paths[i] + f"{other}_bundle_res.npy", allow_pickle=True).tolist())
        other_dicts.append(AICL)
        other_lists.append(AICL)


finetune_models = ["4o-mini", "4.1-mini"]

for baseline in finetune_models:
    for i in range(6):
        responses = load_pkl(os.path.join(six_paths[i], f'finetuned_responses_{baseline}.pkl'))
        results = [extract_json_simple_replace(i) for i in responses]

        other_lists.append(results)

        response_dict = dict(zip(keys[i%3], results))



        other_dicts.append(response_dict)





Extraction error: Extra data: line 7 column 1 (char 159)
Extraction error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Extraction error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Extraction error: Extra data: line 2 column 4 (char 5)
Extraction error: Expecting value: line 3 column 8 (char 10)
Extraction error: Expecting ':' delimiter: line 3 column 5 (char 7)
Extraction error: Expecting ':' delimiter: line 2 column 13 (char 14)
Extraction error: Expecting ',' delimiter: line 3 column 34 (char 86)
Extraction error: Expecting property name enclosed in double quotes: line 5 column 3 (char 60)
Extraction error: Invalid control character at: line 2 column 4 (char 5)
Extraction error: Expecting property name enclosed in double quotes: line 2 column 1 (char 2)
Extraction error: Expecting property name enclosed in double quotes: line 2 column 1 (char 2)
Extraction error: Expecting property name enclosed in double quotes: line 2

# When making bundle strings, do we have to link up the bundlerec and llm4bear sessions to the correct product indices.

In [ ]:
def make_bundle_strings(path, predictions_dict):
    # Load the full ground truth to get ALL keys (the 180)
    test_set = np.load(path + "test_set.npy", allow_pickle=True).tolist()
    all_test_keys = list(test_set.keys())

    new_dict = {}
    bundle_size_dict = {}

    for key in all_test_keys:
        # Get the LLM prediction for this specific key
        value = predictions_dict.get(key)

        session_items = test_set[key].split("|split|")
        empty_box = []
        size_box = []

        # Even if value is None, we should initialize the key in new_dict
        # as an empty list to keep the size at 180
        if value is not None and isinstance(value, dict):
            for bundle_id, content in value.items():
                try:
                    # Use the robust regex-based converter we discussed
                    item_ids = convert_products_to_indices(content)

                    bundle_items = [session_items[j] for j in item_ids if j < len(session_items)]

                    if bundle_items:
                        empty_box.append(bundle_items)
                        size_box.append(len(bundle_items))
                except Exception as e:
                    continue

        # IMPORTANT: Always add the key, even if empty, to maintain N=180
        new_dict[key] = input_strings(empty_box) if empty_box else []
        bundle_size_dict[key] = size_box if size_box else []

    return new_dict, bundle_size_dict

def bundle_numbers(input_dict):
    keys = list(input_dict.keys())

    all_num_bundles = []
    all_bundle_sizes = []
    avg_bundles_per_session = []

    sessions = []

    for i in keys:

        num_bundles = []
        bundle_sizes = []

        turn_dict = input_dict[i]

        actual_keys = list(turn_dict.keys())
        # print(actual_keys[0])

        for j in actual_keys:
            num_bundles.append(len(turn_dict[j]))
            # print(turn_dict[j])
            for k in turn_dict[j]:
                bundle_sizes.append(k)

        bundle_count = sum(num_bundles)
        avg_bundle_size = sum(bundle_sizes) / bundle_count

        all_num_bundles.append(bundle_count)
        all_bundle_sizes.append(avg_bundle_size)
        avg_bundles_per_session.append(bundle_count/len(actual_keys))

        sessions.append(len(actual_keys))

    return all_num_bundles, all_bundle_sizes, avg_bundles_per_session, sessions

def dict_to_list(input_dict):
    keys = list(input_dict.keys())

    bundle_string_list = []

    for i in keys:

        per_key_string_list = []
        run_dict = input_dict[i]

        session_keys = list(run_dict.keys())

        for j in session_keys:
            session_strings = run_dict[j]
            for k in session_strings:
                per_key_string_list.append(k)

        bundle_string_list.append(per_key_string_list)

    return bundle_string_list


def clean_nones(input_list):
    cleaned_list = [item for item in input_list if item is not None]
    return cleaned_list

In [ ]:
DATA_PATHS = {
    'brec_elec': '/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/electronic/',
    'brec_clo': '/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/clothing/',
    'brec_food': '/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/food/',
    'bear_elec': '/content/LLM4BEAR/4_Bundle Generation/data/llm4bear/electronic/',
    'bear_clo': '/content/LLM4BEAR/4_Bundle Generation/data/llm4bear/clothing/',
    'bear_food': '/content/LLM4BEAR/4_Bundle Generation/data/llm4bear/food/'
}


MODEL_LISTS = {
    key: value
    for i in range(len(baseline_names))
    for key, value in [
        (baseline_names[i] + "_zero-shot", [
            ('brec', 'elec', zero_dicts[6*i]), ('brec', 'clo', zero_dicts[6*i+1]), ('brec', 'food', zero_dicts[6*i+2]),
            ('bear', 'elec', zero_dicts[6*i+3]), ('bear', 'clo', zero_dicts[6*i+4]), ('bear', 'food', zero_dicts[6*i+5])
        ]),
        (baseline_names[i] + "_few-shot", [
            ('brec', 'elec', few_dicts[6*i]), ('brec', 'clo', few_dicts[6*i+1]), ('brec', 'food', few_dicts[6*i+2]),
            ('bear', 'elec', few_dicts[6*i+3]), ('bear', 'clo', few_dicts[6*i+4]), ('bear', 'food', few_dicts[6*i+5])
        ])
    ]
}



OTHER_MODEL_LISTS = {
    '4o-mini_AICL': [
        ('brec', 'elec', other_dicts[0]), ('brec', 'clo', other_dicts[1]), ('brec', 'food', other_dicts[2]),
        ('bear', 'elec', other_dicts[3]), ('bear', 'clo', other_dicts[4]), ('bear', 'food', other_dicts[5]),
    ],
    '4.1-mini_AICL': [
        ('brec', 'elec', other_dicts[6]), ('brec', 'clo', other_dicts[7]), ('brec', 'food', other_dicts[8]),
        ('bear', 'elec', other_dicts[9]), ('bear', 'clo', other_dicts[10]), ('bear', 'food', other_dicts[11]),
    ],
    '4o-mini_SFT': [
        ('brec', 'elec', other_dicts[12]), ('brec', 'clo', other_dicts[13]), ('brec', 'food', other_dicts[14]),
        ('bear', 'elec', other_dicts[15]), ('bear', 'clo', other_dicts[16]), ('bear', 'food', other_dicts[17]),
    ],
    '4.1-mini_SFT': [
        ('brec', 'elec', other_dicts[18]), ('brec', 'clo', other_dicts[19]), ('brec', 'food', other_dicts[20]),
        ('bear', 'elec', other_dicts[21]), ('bear', 'clo', other_dicts[22]), ('bear', 'food', other_dicts[23]),
    ]
}


COMBINED_MODELS = MODEL_LISTS | OTHER_MODEL_LISTS


# 3. Initialize Result Dictionaries
new_string_dicts = {}
size_dicts = {}


# --- 4. Single Comprehensive Loop (Fixes KeyError) ---

for model_type, items in COMBINED_MODELS.items():
    for source, category, data_list in items:

        # --- 1. Correct Path Mapping ---
        # We must combine source ('brec'/'bear') with category ('elec'/'clo'/'food')
        path_key = f"{source}_{category}"
        data_path = DATA_PATHS.get(path_key)

        if not data_path:
            print(f"⚠️ Warning: No path found for {path_key}. Skipping...")
            continue



        # Extract base model name
        base_model = model_type
        key_name = f"{base_model}_{source}_{category}"

        # --- 4. Execute and Store ---
        print(f"🔄 Processing: {key_name}")
        print(f"📂 Path: {data_path}")

        new_string_dict, size_dict = make_bundle_strings(data_path, data_list)

        new_string_dicts[key_name] = new_string_dict
        size_dicts[key_name] = size_dict

print("\n--- Automation Complete ---")
print(f"Total processed results stored: {len(new_string_dicts)}")


🔄 Processing: gpt-4o-mini_zero-shot_brec_elec
📂 Path: /content/LLM4BEAR/4_Bundle Generation/data/bundlerec/electronic/
🔄 Processing: gpt-4o-mini_zero-shot_brec_clo
📂 Path: /content/LLM4BEAR/4_Bundle Generation/data/bundlerec/clothing/
🔄 Processing: gpt-4o-mini_zero-shot_brec_food
📂 Path: /content/LLM4BEAR/4_Bundle Generation/data/bundlerec/food/
🔄 Processing: gpt-4o-mini_zero-shot_bear_elec
📂 Path: /content/LLM4BEAR/4_Bundle Generation/data/llm4bear/electronic/
🔄 Processing: gpt-4o-mini_zero-shot_bear_clo
📂 Path: /content/LLM4BEAR/4_Bundle Generation/data/llm4bear/clothing/
🔄 Processing: gpt-4o-mini_zero-shot_bear_food
📂 Path: /content/LLM4BEAR/4_Bundle Generation/data/llm4bear/food/
🔄 Processing: gpt-4o-mini_few-shot_brec_elec
📂 Path: /content/LLM4BEAR/4_Bundle Generation/data/bundlerec/electronic/
🔄 Processing: gpt-4o-mini_few-shot_brec_clo
📂 Path: /content/LLM4BEAR/4_Bundle Generation/data/bundlerec/clothing/
🔄 Processing: gpt-4o-mini_few-shot_brec_food
📂 Path: /content/LLM4BEAR/4_B

In [ ]:
def generate_bundle_stats(input_dict):

    avg_num_bundles_per_sess, avg_num_items_per_bundle, num_bundles_generated, session_count = [], [], [], []

    for run_keys in list(input_dict.keys()):

        num_bundles = []
        bundle_sizes = []
        num_sessions = []

        count = 0

        run_dict = input_dict[run_keys]



        test_keys = list(run_dict.keys())


        for j in test_keys:

            values = run_dict[j]

            if values and len(values) > 0:

                num_bundles.append(len(values))

                for i in values:
                    bundle_sizes.append(i)

                count += 1

        avg_num_bundles_per_sess.append(sum(num_bundles) / count)
        avg_num_items_per_bundle.append(sum(bundle_sizes) / sum(num_bundles))
        session_count.append(count)
        num_bundles_generated.append(sum(num_bundles))


    return avg_num_bundles_per_sess, avg_num_items_per_bundle, num_bundles_generated, session_count


In [ ]:
with open("/content/drive/My Drive/evaluating_bundles/all_baseline_bundle_strings.pkl", "wb") as f:
    pickle.dump(new_string_dicts, f)

with open("/content/drive/My Drive/evaluating_bundles/all_baseline_bundle_sizes.pkl", "wb") as f:
    pickle.dump(size_dicts, f)


# 1. Define the desired dictionary keys
keys = [
    "num_generated_bundles",
    "avg_bundle_sizes",
    "avg_num_bundles_per_sess",
    "num_session"
]

# 2. Define the corresponding list variables (must be in the same order as the keys)
values = generate_bundle_stats(size_dicts)

# 3. Create the dictionary using zip()
results_dict = dict(zip(keys, values))

with open("/content/drive/My Drive/evaluating_bundles/all_baseline_bundle_stats.pkl", "wb") as f:
    pickle.dump(results_dict, f)

In [49]:
with open("/content/drive/My Drive/evaluating_bundles/all_baseline_bundle_strings.pkl", "rb") as f:
    baseline_generated_bundle_strings = pickle.load(f)

with open("/content/drive/My Drive/evaluating_bundles/all_baseline_bundle_sizes.pkl", "rb") as f:
    baseline_generated_bundle_sizes = pickle.load(f)

with open("/content/drive/My Drive/evaluating_bundles/all_baseline_bundle_stats.pkl", "rb") as f:
    baseline_generated_bundle_stats = pickle.load(f)

In [50]:
baseline_testing_dict = dict_to_list(baseline_generated_bundle_strings)

baseline_key_names = list(baseline_generated_bundle_strings.keys())

print(baseline_key_names)
print(len(baseline_key_names))
print()


stat_keys = ["num_generated_bundles",
    "avg_bundle_sizes",
    "avg_num_bundles_per_sess",
    "num_session"
]

avg_bundle_per_sess = baseline_generated_bundle_stats[stat_keys[0]]
bundle_sizes = baseline_generated_bundle_stats[stat_keys[1]]
num_bundles = baseline_generated_bundle_stats[stat_keys[2]]
num_sessions = baseline_generated_bundle_stats[stat_keys[3]]

for i in range(len(baseline_key_names)):

    print(baseline_key_names[i])
    print(f"{num_sessions[i]} Sessions, Total Bundles Generated: {num_bundles[i]}")
    print(f"Average Bundles per Session: {avg_bundle_per_sess[i]:.3f}, Average Bundle Size: {bundle_sizes[i]:.3f}")
    print()

['gpt-4o-mini_zero-shot_brec_elec', 'gpt-4o-mini_zero-shot_brec_clo', 'gpt-4o-mini_zero-shot_brec_food', 'gpt-4o-mini_zero-shot_bear_elec', 'gpt-4o-mini_zero-shot_bear_clo', 'gpt-4o-mini_zero-shot_bear_food', 'gpt-4o-mini_few-shot_brec_elec', 'gpt-4o-mini_few-shot_brec_clo', 'gpt-4o-mini_few-shot_brec_food', 'gpt-4o-mini_few-shot_bear_elec', 'gpt-4o-mini_few-shot_bear_clo', 'gpt-4o-mini_few-shot_bear_food', 'gpt-4.1-mini_zero-shot_brec_elec', 'gpt-4.1-mini_zero-shot_brec_clo', 'gpt-4.1-mini_zero-shot_brec_food', 'gpt-4.1-mini_zero-shot_bear_elec', 'gpt-4.1-mini_zero-shot_bear_clo', 'gpt-4.1-mini_zero-shot_bear_food', 'gpt-4.1-mini_few-shot_brec_elec', 'gpt-4.1-mini_few-shot_brec_clo', 'gpt-4.1-mini_few-shot_brec_food', 'gpt-4.1-mini_few-shot_bear_elec', 'gpt-4.1-mini_few-shot_bear_clo', 'gpt-4.1-mini_few-shot_bear_food', 'gemini_zero-shot_brec_elec', 'gemini_zero-shot_brec_clo', 'gemini_zero-shot_brec_food', 'gemini_zero-shot_bear_elec', 'gemini_zero-shot_bear_clo', 'gemini_zero-shot_b

In [ ]:

def compute_comprehensively(ground_indices, predictions):
    session_precision = 0
    session_recall = 0
    session_jaccard = 0
    invalid_id = []
    valid_session_count = 0

    for test_id, pred in predictions.items():
        all_bundle = ground_indices[test_id]

        hit_bundle = 0


        if pred is None or isinstance(pred, int) or len(pred) == 0:
            invalid_id.append(test_id)
            continue

        valid_session_count += 1

        # 2. Convert all predicted bundles into sets of indices
        pred_sets = []
        for bundle_key, product_nums in pred.items():
            # e.g., converts ["product1", "product2"] to {0, 1}
            indices = convert_products_to_indices(product_nums)
            if indices and len(indices) >= 2:
                pred_sets.append(set(indices))
            # if len(indices) == 1:
            #     anti_reward += -0.15

        gt_sets = [set(gt) for gt in all_bundle]

        # 3. Greedy Matching: Assign the best prediction to each GT
        all_matches = []
        for g_idx, g_set in enumerate(gt_sets):
            for p_idx, p_set in enumerate(pred_sets):
                intersection = len(g_set & p_set)
                union = len(g_set | p_set)
                score = intersection / union if union > 0 else 0
                if score > 0:
                    all_matches.append((score, g_idx, p_idx))

        # Sort by best Jaccard score first
        all_matches.sort(key=lambda x: x[0], reverse=True)

        assigned_gts = set()
        assigned_ps = set()
        reward_sum = 0

        for score, g_idx, p_idx in all_matches:
            if g_idx not in assigned_gts and p_idx not in assigned_ps:
                assigned_gts.add(g_idx)
                assigned_ps.add(p_idx)
                reward_sum += score

        # Final Reward: Average Jaccard across all Ground Truths
        # (This forces the model to cover ALL ground truth bundles, not just one)
        session_jaccard += reward_sum / max(len(gt_sets), 1*len(pred_sets))


        for truth_bundle in gt_sets:
            for pred_bundle in pred_sets:
                if set(pred_bundle) == truth_bundle:
                    hit_bundle += 1



        session_precision += hit_bundle / len(pred_sets) if len(pred_sets) > 0 else 0
        session_recall += hit_bundle / len(gt_sets) if len(gt_sets) > 0 else 0

    session_precision /= valid_session_count
    session_recall /= valid_session_count
    session_jaccard /= valid_session_count

    return session_precision, session_recall, session_jaccard, list(set(invalid_id))



def path_compute_comprehensively(path, results, inbuilt = "no"):

    test_set_path = "test_set.npy"
    test_set = np.load(path + test_set_path, allow_pickle=True).tolist()


    test_ground_indices_path = "test_ground_indices.pkl"

    with open(path + test_ground_indices_path, 'rb') as f:
        ground_indices_test = pkl.load(f)


    if inbuilt == "no":
        test_keys = list(test_set.keys())

        # print(test_keys)

        results_dict = {
        key: result
        for key, result in zip(test_keys, results)
        }

        session_precision, session_recall, session_jaccard, invalids = compute_comprehensively(ground_indices_test, results_dict)
    else:
        session_precision, session_recall, session_jaccard, invalids = compute_comprehensively(ground_indices_test, results)

    # print(path)
    print(f'Precision: {session_precision:.3f}, Recall: {session_recall:.3f}, Jaccard: {session_jaccard:.3f}')
    print()

    return session_precision, session_recall, session_jaccard, invalids




In [55]:
electronic_bundlerec_loc = '/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/electronic/'
clothing_bundlerec_loc = '/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/clothing/'
food_bundlerec_loc = '/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/food/'

electronic_llm4bear_loc = '/content/LLM4BEAR/4_Bundle Generation/data/llm4bear/electronic/'
clothing_llm4bear_loc = '/content/LLM4BEAR/4_Bundle Generation/data/llm4bear/clothing/'
food_llm4bear_loc = '/content/LLM4BEAR/4_Bundle Generation/data/llm4bear/food/'

six_locs = [electronic_bundlerec_loc, clothing_bundlerec_loc, food_bundlerec_loc,
            electronic_llm4bear_loc, clothing_llm4bear_loc, food_llm4bear_loc]


precision_list = []
recall_list = []
jaccard_list = []

count = 0

for i in range(6):
    for j in range(6):
        print(baseline_key_names[count])
        precision, recall, jaccard,_ = path_compute_comprehensively(six_locs[i], zero_lists[j])
        count += 1

        precision_list.append(precision)
        recall_list.append(recall)
        jaccard_list.append(jaccard)


    for j in range(6):
        print(baseline_key_names[count])
        precision, recall, jaccard,_ = path_compute_comprehensively(six_locs[i], few_lists[j])
        count += 1

        precision_list.append(precision)
        recall_list.append(recall)
        jaccard_list.append(jaccard)

for i in range(12):
    print(baseline_key_names[count])
    precision, recall, jaccard,_ = path_compute_comprehensively(six_locs[i%6], other_lists[i], inbuilt="yes")
    count += 1

    precision_list.append(precision)
    recall_list.append(recall)
    jaccard_list.append(jaccard)

for i in range(12,24):
    print(baseline_key_names[count])
    precision, recall, jaccard,_ = path_compute_comprehensively(six_locs[i%6], other_lists[i])
    count += 1

    precision_list.append(precision)
    recall_list.append(recall)
    jaccard_list.append(jaccard)


gpt-4o-mini_zero-shot_brec_elec
Precision: 0.428, Recall: 0.400, Jaccard: 0.558

gpt-4o-mini_zero-shot_brec_clo
Precision: 0.098, Recall: 0.089, Jaccard: 0.326

gpt-4o-mini_zero-shot_brec_food
Precision: 0.085, Recall: 0.106, Jaccard: 0.323

gpt-4o-mini_zero-shot_bear_elec
Precision: 0.298, Recall: 0.279, Jaccard: 0.479

gpt-4o-mini_zero-shot_bear_clo
Precision: 0.074, Recall: 0.066, Jaccard: 0.309

gpt-4o-mini_zero-shot_bear_food
Precision: 0.061, Recall: 0.064, Jaccard: 0.303

gpt-4o-mini_few-shot_brec_elec
Precision: 0.443, Recall: 0.409, Jaccard: 0.561

gpt-4o-mini_few-shot_brec_clo
Precision: 0.100, Recall: 0.088, Jaccard: 0.339

gpt-4o-mini_few-shot_brec_food
Precision: 0.075, Recall: 0.099, Jaccard: 0.334

gpt-4o-mini_few-shot_bear_elec
Precision: 0.299, Recall: 0.277, Jaccard: 0.502

gpt-4o-mini_few-shot_bear_clo
Precision: 0.094, Recall: 0.083, Jaccard: 0.309

gpt-4o-mini_few-shot_bear_food
Precision: 0.071, Recall: 0.069, Jaccard: 0.321

gpt-4.1-mini_zero-shot_brec_elec
Preci

# You can evaluate the bundles if you want to double-check the LLM-scores from the paper. Obviously as it's non-deterministic, the number will be slightly different, but the general trends will be the same.

In [45]:
baseline_key_names

['gpt-4o-mini_zero-shot_brec_elec',
 'gpt-4o-mini_zero-shot_brec_clo',
 'gpt-4o-mini_zero-shot_brec_food',
 'gpt-4o-mini_zero-shot_bear_elec',
 'gpt-4o-mini_zero-shot_bear_clo',
 'gpt-4o-mini_zero-shot_bear_food',
 'gpt-4o-mini_few-shot_brec_elec',
 'gpt-4o-mini_few-shot_brec_clo',
 'gpt-4o-mini_few-shot_brec_food',
 'gpt-4o-mini_few-shot_bear_elec',
 'gpt-4o-mini_few-shot_bear_clo',
 'gpt-4o-mini_few-shot_bear_food',
 'gpt-4.1-mini_zero-shot_brec_elec',
 'gpt-4.1-mini_zero-shot_brec_clo',
 'gpt-4.1-mini_zero-shot_brec_food',
 'gpt-4.1-mini_zero-shot_bear_elec',
 'gpt-4.1-mini_zero-shot_bear_clo',
 'gpt-4.1-mini_zero-shot_bear_food',
 'gpt-4.1-mini_few-shot_brec_elec',
 'gpt-4.1-mini_few-shot_brec_clo',
 'gpt-4.1-mini_few-shot_brec_food',
 'gpt-4.1-mini_few-shot_bear_elec',
 'gpt-4.1-mini_few-shot_bear_clo',
 'gpt-4.1-mini_few-shot_bear_food',
 'gemini_zero-shot_brec_elec',
 'gemini_zero-shot_brec_clo',
 'gemini_zero-shot_brec_food',
 'gemini_zero-shot_bear_elec',
 'gemini_zero-shot_be

In [ ]:
super_prompts = [electronic_prompts, clothing_prompts, food_prompts]

for i in range(96):

    # await intent_evaluation_module(charizards=[f"{baseline_key_names[i]}_bad", f"{baseline_key_names[i]}_middle", f"{baseline_key_names[i]}_good"],
    #                                                    prompts=super_prompts[int(i % 3)],
    #                                                    input_strings=baseline_testing_dict[i])

In [47]:
quality = ["bad", "middle", "good"]

baseline_scores = []

for i in range(96):

    run_scores = []

    for j in range(3):


        # filename = f"/content/drive/MyDrive/evaluating_bundles/intent_{baseline_key_names[i]}_{quality[j]}.pkl"
        filename = f'/content/LLM4BEAR/4_Bundle Generation/evaluation/intent_{baseline_key_names[i]}_{quality[j]}.pkl'

        with open(filename, 'rb') as f:
            responses, verdicts, scores = pickle.load(f)

        part_scores = clean_nones(scores)

        # print(scores)

        run_scores.append(sum(part_scores)/len(part_scores))


    baseline_scores.append([run_scores, sum(run_scores)/3])

for i in range(96):
    print(f"intent_{baseline_key_names[i]}, Average Score: {baseline_scores[i][1]:.3f}")#, All Scores: {all_scores[i][0]}")

    if i % 3 == 2:

        print()

intent_gpt-4o-mini_zero-shot_brec_elec, Average Score: 3.208
intent_gpt-4o-mini_zero-shot_brec_clo, Average Score: 3.692
intent_gpt-4o-mini_zero-shot_brec_food, Average Score: 3.564

intent_gpt-4o-mini_zero-shot_bear_elec, Average Score: 3.322
intent_gpt-4o-mini_zero-shot_bear_clo, Average Score: 3.851
intent_gpt-4o-mini_zero-shot_bear_food, Average Score: 3.708

intent_gpt-4o-mini_few-shot_brec_elec, Average Score: 3.255
intent_gpt-4o-mini_few-shot_brec_clo, Average Score: 3.755
intent_gpt-4o-mini_few-shot_brec_food, Average Score: 3.594

intent_gpt-4o-mini_few-shot_bear_elec, Average Score: 3.521
intent_gpt-4o-mini_few-shot_bear_clo, Average Score: 3.844
intent_gpt-4o-mini_few-shot_bear_food, Average Score: 3.757

intent_gpt-4.1-mini_zero-shot_brec_elec, Average Score: 3.609
intent_gpt-4.1-mini_zero-shot_brec_clo, Average Score: 4.037
intent_gpt-4.1-mini_zero-shot_brec_food, Average Score: 3.805

intent_gpt-4.1-mini_zero-shot_bear_elec, Average Score: 3.847
intent_gpt-4.1-mini_zero-s

In [56]:
import pandas as pd

# 1. Prepare a list to hold our formatted rows
formatted_data = []

# 2. Loop through the 96 entries (matching your print loop)
for i in range(96):
    # print(i)
    full_title = baseline_key_names[i]

    # Optional: Split the title into parts for better CSV columns
    # Example: "gpt41_bear_clo_zero" -> ["gpt41", "bear", "clo", "zero"]
    parts = full_title.split('_')

    model = parts[0] if len(parts) > 0 else "N/A"
    dataset = parts[1] if len(parts) > 1 else "N/A"
    category = parts[2] if len(parts) > 2 else "N/A"
    run_type = parts[3] if len(parts) > 3 else "N/A"


    # 3. Create a dictionary for this row
    row = {
        "Title": full_title,
        "Model": model,
        "Dataset": dataset,
        "Category": category,
        "Run_Type": run_type,
        "Sessions": num_sessions[i], # sometimes the LLM gives a poor response, so we just remove it, llama is one instance, which may explain why its LLM scores are higher than other baselines.
        "Total_Bundles": num_bundles[i],
        "Avg_Bundles_Per_Sess": avg_bundle_per_sess[i],
        "Avg_Bundle_Size": bundle_sizes[i],
        "Precision": precision_list[i],
        "Recall": recall_list[i],
        "Jaccard": jaccard_list[i],
        "Average_Score": baseline_scores[i][1]
    }

    formatted_data.append(row)

# 4. Create DataFrame
df_stats = pd.DataFrame(formatted_data)

# 5. Export to CSV
df_stats.to_csv("baseline_bundle_stats.csv", index=False)

print("CSV saved successfully as 'baseline_bundle_stats.csv'")

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
CSV saved successfully as 'baseline_bundle_stats.csv'
